# Dataset study

This notebook studies the **built** dataset (`built/` of the dataset folder, the same one published on Hugging Face): the arrays the
model will be trained and tested on. It looks at one application at a time: set `APP` in the parameter cell and run
everything again.

## How to read this notebook

Every section follows the same pattern:

1. **Question**: what the section wants to find out.
2. **What is plotted**: every element of the chart: axes and units, what one cell, line or box is, what the colours,
   grey areas and dashed lines mean.
3. The chart.
4. **How to read it**: numbers printed from the data, with a worked example on one cell. They are recomputed for the
   `APP` you choose.
5. **Reading**: what the section found on the build of 2026-11-16 (builder 1533dfa), for Hotel and for Social, and what it
   means for the model.

## How a run unfolds

```
Fault runs (dev, test; noop runs inject nothing):
  t_workload                 t_inject                                     end
      |----- settle, 300 s -----|----------- observation, 900 s -----------|
  wrk2 load starts         fault injected

Clean sessions (training):
  t_workload                 t_inject                                                            end
      |----- settle, 300 s -----|---------------------- observation, 5400 s ----------------------|
  wrk2 load starts         nothing injected (t_inject only marks the same point in time)
```

- The data sit on a grid of 30 s steps. The value at time `t` summarises the interval (t − 30 s, t].
- `t_rel` is the time in seconds from the injection: negative during the settle.
- Steps before `t_workload` have no load: there `rate` and `mix` are −1.
- The wrk2 load runs until the end in every run, following its schedule of rates and mixes.

## Splits

| split | runs per app | use |
|---|---|---|
| `clean` | 12 healthy sessions, 90 minutes each | training |
| `dev` | 2 per fault scenario; the faults differ from the test ones | tuning: threshold, window, latent size |
| `test` | 5 per scenario, the fault injected at 10, 20, 40, 80 and 160 req/s; `noop` injects nothing | final evaluation only |

## Glossary

- **steady step**: a step whose load (rate and mix) has not changed for at least 60 s.
- **load segment**: a stretch of constant rate and mix in the wrk2 schedule. Ramps are chains of 30 s segments.
- **mix**: which kinds of request wrk2 sends, the table in section 0 lists them.
- **robust z**: (value − median) / IQR, with median and IQR taken on a reference the settle, or the clean sessions.
- **Spearman ρ**: the correlation of the ranks of two signals: +1 when one always rises with the other, 0 when they are
  unrelated, −1 when one falls as the other rises.
- **PCA, k95**: principal component analysis finds the independent directions along which a service's features move
  together. k95 is how many directions explain 95% of the variance: the effective number of dimensions of the data.
- **autocorrelation at lag k**: the correlation between a signal at step t and the same signal k steps later. 1 means
  the signal barely changes in k steps; 0 means it keeps no memory.
- **symlog axis**: linear near 0 and logarithmic further out, so small and huge values fit in one chart.

In [ ]:
import json
import sys
import warnings
from pathlib import Path
from types import SimpleNamespace

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from anomaly import build
from anomaly import explore as ex

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 110, "display.max_rows", 80)

# or: from huggingface_hub import snapshot_download
#     ROOT = Path(snapshot_download("Keeevin16/AIOpsLab-app-collection-Hotel-Social", repo_type="dataset",
#                                   allow_patterns=["built/*/*/*.npz", "built/*/*/*.json"])) / "built"
ROOT = build.DATASET / "built"
APP = "hotel"       # "hotel" or "social"
RUN = None          # run of section 1, e.g. "test/network_delay_hotel_res__r1"; None = the first test run
SPLIT = "dev"       # fault runs of section 6: "dev" or "test" (look at test only for the final evaluation)
MEM_DELTA = False   # True: in sections 4-5 memory becomes its change per step (MiB) instead of its level

## 0 · Loading

**What gets loaded.**

- `runs`: every run of the app, keyed `split/name`. Each run has the same fields as `explore.load`, so the `explore`
  helpers (`robust_z`, `settle_mask`, `steady_mask`, `target_services`, `score_services`) work here too.
  - `x [T, S, F]`: the value per step, service and feature; NaN where no sample fell in the step.
  - `t`, `t_rel`, `rate`, `mix`: per step.
  - `labels`: the run's `.json`: timeline, fault, target, load segments, checks.
- `info`: one row per run.
- `df`: the whole app in long format. There is one row per (run, step, service) and one column per feature, plus:
  - `rate` and `mix` of the step;
  - `split`;
  - `phase`: `off` without load, `before` up to the injection, `after` after it;
  - `steady`: True on a steady step.

  This is the table to explore with `groupby` and `query`.
- `prep(run)`: the features as the model would see them: CPU in millicores, memory in MiB, restarts per 5 minutes,
  then `log1p`. `log1p(v) = log(1 + v)` compresses large values while keeping 0 at 0.
- `Z`: the `prep` values standardised with the mean and standard deviation of the clean sessions, per service and
  feature. 0 = the clean average, ±1 = one standard deviation away.

The table printed below lists the runs per split and scenario. `rate_inj` is the wrk2 rate at the injection step of
each run, `minutes` is the length of a run and `target` is the faulty service.

In [ ]:
def load_built(npz):
    z, labels = np.load(npz), json.load(open(npz.with_suffix(".json")))
    run = SimpleNamespace(**{k: z[k] for k in z.files}, labels=labels)
    run.services, run.features = [str(s) for s in run.services], [str(f) for f in run.features]
    run.split, run.id, run.kind = npz.parent.name, f"{npz.parent.name}/{npz.stem}", labels["kind"]
    run.t_rel = run.t - labels["t_inject"]
    return run


def scenario(r):
    return r.id.split("/")[1].rsplit("__", 1)[0].removesuffix("_hotel_res").removesuffix("_social_network")


def rate_at_injection(r):
    return int(r.rate[min(np.searchsorted(r.t, r.labels["t_inject"]), len(r.t) - 1)])


runs = {r.id: r for r in map(load_built, sorted((ROOT / APP).glob("*/*.npz")))}
first = next(iter(runs.values()))
SERVICES, FEATURES, STEP = first.services, first.features, first.labels["step"]
assert all(r.services == SERVICES and r.features == FEATURES for r in runs.values())
clean_ids = [k for k, r in runs.items() if r.split == "clean"]

info = pd.DataFrame([{
    "run": r.id, "split": r.split, "scenario": scenario(r), "rep": r.labels["rep"], "fault": r.labels["fault_type"],
    "target": ", ".join(ex.target_services(r)) or "-", "rate_inj": rate_at_injection(r), "steps": len(r.t),
    "minutes": round((r.t[-1] - r.t[0]) / 60),
} for r in runs.values()]).set_index("run")


def frame(runs):
    parts = []
    for r in runs.values():
        T, S, F = r.x.shape
        index = pd.MultiIndex.from_arrays([np.repeat(r.id, T * S), np.repeat(r.t_rel, S), np.tile(r.services, T)],
                                          names=["run", "t_rel", "service"])
        part = pd.DataFrame(r.x.reshape(T * S, F).astype(np.float64), index=index, columns=r.features)
        phase = np.where(r.rate < 0, "off", np.where(r.t_rel <= 0, "before", "after"))
        part = part.assign(rate=np.repeat(r.rate, S), mix=np.repeat(r.mix, S), split=r.split,
                           phase=np.repeat(phase, S), steady=np.repeat(ex.steady_mask(r), S))
        parts.append(part)
    return pd.concat(parts)


df = frame(runs)
clean = df[(df.split == "clean") & (df.rate >= 0)]
steady_clean = clean[clean.steady]

UNIT = {"cpu": 1e3, "cpu_user": 1e3, "cpu_system": 1e3, "cpu_throttled": 1e3,  # millicores
        "mem_working_set": 2**-20, "mem_rss": 2**-20, "mem_cache": 2**-20,     # MiB
        "restarts": 300}                                                          # per 5 minutes
MEMORY = [f for f in FEATURES if f.startswith("mem_")]


def prep(r, mem_delta=MEM_DELTA):
    v = np.clip(r.x, 0, None) * np.array([UNIT.get(f, 1) for f in FEATURES])
    out = np.log1p(v)
    if mem_delta:
        for f in MEMORY:
            j = FEATURES.index(f)
            d = np.diff(v[:, :, j], axis=0, prepend=np.nan)
            out[:, :, j] = np.sign(d) * np.log1p(np.abs(d))
    return out


P = {k: prep(r) for k, r in runs.items()}
ref = np.concatenate([P[k][runs[k].rate >= 0] for k in clean_ids])
MU, SD = np.nanmean(ref, axis=0), np.nanstd(ref, axis=0)
Z = {k: (v - MU) / np.where(SD > 1e-6, SD, np.nan) for k, v in P.items()}


def show(v, f):
    # a raw value in a readable unit
    if v is None or np.isnan(v):
        return "NaN"
    if f.startswith("cpu"):
        return f"{v * 1e3:.2f} mC"
    if f.startswith("mem_"):
        return f"{v / 2**20:.1f} MiB" if abs(v) >= 2**20 else f"{v / 2**10:.1f} KiB"
    if f.endswith("_bytes"):
        return f"{v / 1e3:.2f} kB/s"
    if f.endswith("_packets"):
        return f"{v:.1f} pkt/s"
    if f == "restarts":
        return f"{v * 300:.2f} per 5 min"
    return f"{v:.1f}"


def heat(data, title, cbar, fmt="{:.2f}", cmap="RdBu_r", vmin=-1, vmax=1, annot=True, ax=None,
         xlabel="feature", ylabel="service"):
    # annotated heatmap of a DataFrame (rows = index, columns = columns) with a labelled colour bar
    if ax is None:
        _, ax = plt.subplots(figsize=(2.5 + .6 * data.shape[1], 1.2 + .3 * data.shape[0]), layout="constrained")
    a = data.to_numpy(dtype=float)
    im = ax.imshow(a, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.figure.colorbar(im, ax=ax, label=cbar, shrink=.8)
    ax.set_xticks(range(data.shape[1]), [str(c) for c in data.columns], rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(data.shape[0]), [str(i) for i in data.index], fontsize=7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if annot:
        for (i, k), v in np.ndenumerate(a):
            if not np.isnan(v):
                p = (v - vmin) / (vmax - vmin)
                dark = p > .6 if cmap == "Reds" else (p < .2 or p > .8)
                ax.text(k, i, fmt.format(v), ha="center", va="center", fontsize=6, color="white" if dark else "black")
    ax.set_title(title, fontsize=10)
    return ax


def extremes(data, n=3, fmt="{:.2f}"):
    # the n highest and lowest cells of a services x features table
    s = data.stack().dropna().sort_values()
    line = lambda part: ", ".join(f"{i[0]}/{i[1]} {fmt.format(v)}" for i, v in part.items())
    print(f"highest: {line(s.tail(n)[::-1])}\nlowest:  {line(s.head(n))}")


print(f"{APP}: {len(runs)} runs, {len(SERVICES)} services, {len(FEATURES)} features; df has {len(df)} rows")
info.groupby(["split", "scenario"]).agg(runs=("rep", "size"), rate_inj=("rate_inj", lambda v: sorted(v)),
                                        minutes=("minutes", "median"), target=("target", "first"))

### Features, services and request mixes

Three reference tables:

- **Features.** For each feature: the Prometheus metric it comes from, and whether it is a counter turned into a rate
  per second or a gauge taken as is. Also: where it is measured, its raw unit, the unit after `prep`, and what it means.
- **Services.** For each service (the S axis): its type in the datagraph (Service, Database or Cache), who calls it,
  and what it calls or uses. A fault in one service shows up in its callers too: this is the map to read
  localization results.
- **Request mixes.** What wrk2 sends under each `mix` value, taken from the wrk2 payload script.

In [ ]:
MEANING = {
    "cpu": "CPU time used, all modes; 1 core = one CPU fully busy",
    "cpu_user": "CPU time in user space: the application's own code",
    "cpu_system": "CPU time in the kernel for the container: system calls, network stack, I/O",
    "cpu_throttled": "time the container was held back by its CPU limit (only where a limit exists: Hotel)",
    "mem_working_set": "memory in use as Kubernetes counts it (RSS plus recently used page cache); used for OOM kills",
    "mem_rss": "anonymous memory: heap and stacks, not backed by files",
    "mem_cache": "page cache: file data kept in memory (for a database, its data files)",
    "threads": "number of threads in the container",
    "rx_bytes": "bytes received per second on the pod's network interface",
    "tx_bytes": "bytes sent per second on the pod's network interface",
    "rx_packets": "packets received per second",
    "tx_packets": "packets sent per second",
    "restarts": "container restarts per second (kube-state-metrics); non-zero only around a restart",
}
RAW_UNIT = {"cpu": "cores", "cpu_user": "cores", "cpu_system": "cores", "cpu_throttled": "s per s", "threads": "count",
            "mem_working_set": "bytes", "mem_rss": "bytes", "mem_cache": "bytes", "rx_bytes": "B/s", "tx_bytes": "B/s",
            "rx_packets": "packets/s", "tx_packets": "packets/s", "restarts": "per s"}
PREP_UNIT = {"cores": "log1p(mC)", "s per s": "log1p(mC)", "bytes": "log1p(MiB)", "per s": "log1p(per 5 min)"}
WHERE = {"container": "app containers of the pod, summed", "pod": "the pod's network interface",
         "kube": "kube-state-metrics"}
features_table = pd.DataFrame([{
    "feature": name, "Prometheus metric": metric, "kind": "counter -> rate" if kind == "rate" else "gauge",
    "measured on": WHERE[level], "raw unit": RAW_UNIT[name],
    "after prep": PREP_UNIT.get(RAW_UNIT[name], f"log1p({RAW_UNIT[name]})"), "meaning": MEANING[name],
} for name, metric, kind, level in build.FEATURES]).set_index("feature")
display(features_table)

GRAPH = REPO / "service-graph" / build.DATAGRAPH[first.labels["app_scenario"]]
NODE_TYPE = dict(build.datagraph_nodes(GRAPH))
EDGES = build.datagraph_edges(GRAPH)
services_table = pd.DataFrame({
    "type": [NODE_TYPE[s] for s in SERVICES],
    "called by": [", ".join(a for a, _, b in EDGES if b == s) or "-" for s in SERVICES],
    "calls or uses": [", ".join(b for a, _, b in EDGES if a == s) or "-" for s in SERVICES],
}, index=pd.Index(SERVICES, name="service"))
display(services_table)

MIXES = {
    "hotel": {0: ("default", "60% search, 39% recommend, 0.5% login, 0.5% reserve"),
              1: ("search-heavy", "80% search, 19% recommend"),
              2: ("recommend-heavy", "40% search, 59% recommend"),
              3: ("bookings", "login and reserve 5% each: 10x the traffic of user and reservation")},
    "social": {0: ("compose only", "100% compose post"),
               1: ("some reads", "75% compose, 17% home timeline, 8% user timeline"),
               2: ("more reads", "50% compose, 33% home timeline, 17% user timeline")},
}[APP]
pd.DataFrame(MIXES, index=["name", "requests"]).T.rename_axis("mix")

## 1 · One run

**Question.** What happens in a run, from the start of the load to the end?

**What is plotted.**

- **Top chart: load and phases.**
  - The x-axis is minutes from the injection (0 = injection).
  - Blue step line: the wrk2 target rate (req/s, left axis). Orange step line: the request mix (right axis).
  - Shaded bands: grey = no load yet, blue = settle (the run's reference of normal), red = after the injection.
  - Red dashed line: the injection command (`t_inject`). Red dotted line: the moment the injector reported the fault
    active (`t_injected`).
- **Heatmaps: one panel per feature.**
  - Each row is a service and each column a 30 s step.
  - Colour = robust z against the settle of this same run, clipped at ±8. White means as in the settle, red higher,
    blue lower. Grey means no sample in that step.
  - Because the reference is the settle, a load change after the settle colours every feature that follows the load,
    even without a fault. This is the load confound the model must learn to ignore.
- **Table: one row per wrk2 segment.** Start (minutes from the injection), length, target rate and mix, then what wrk2
  delivered: rate, p99 latency and non-2xx (failed) responses.

In [ ]:
r = runs[RUN] if RUN else next(v for k, v in runs.items() if k.startswith("test/"))
minutes = r.t_rel / 60
rel = lambda t: (t - r.labels["t_inject"]) / 60
print(f"{r.id} | fault: {r.labels['fault_type']} | target: {', '.join(ex.target_services(r)) or '-'}"
      f" | rate at the injection: {rate_at_injection(r)} req/s")

fig, ax = plt.subplots(figsize=(13, 2.8), layout="constrained")
ax.axvspan(minutes[0] - STEP / 60, rel(r.labels["t_workload"]), color=".88", label="no load")
ax.axvspan(rel(r.labels["t_workload"]), 0, color="tab:blue", alpha=.12, label="settle (reference)")
ax.axvspan(0, minutes[-1], color="tab:red", alpha=.07, label="after the injection")
ax.step(minutes, np.where(r.rate >= 0, r.rate, np.nan), where="pre", color="tab:blue", label="wrk2 rate")
ax.axvline(0, color="red", ls="--", label="injection (t_inject)")
ax.axvline(rel(r.labels["t_injected"]), color="red", ls=":", label="fault active (t_injected)")
ax.set_xlabel("minutes from the injection")
ax.set_ylabel("target rate (req/s)")
ax2 = ax.twinx()
ax2.step(minutes, np.where(r.mix >= 0, r.mix, np.nan), where="pre", color="tab:orange", alpha=.8, label="mix")
ax2.set_yticks(sorted(MIXES), [f"{m}: {MIXES[m][0]}" for m in sorted(MIXES)], fontsize=7)
ax2.set_ylim(-.5, max(MIXES) + .5)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
fig.legend(h1 + h2, l1 + l2, fontsize=7, ncol=7, loc="outside lower center")
ax.set_title(f"{r.id}: load and phases", fontsize=10)
plt.show()

zr = ex.robust_z(r.x, ex.settle_mask(r))
cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad("#9a9a9a")
fig, axes = plt.subplots(4, 4, figsize=(16, 3 + .5 * len(SERVICES)), sharex=True, sharey=True, layout="constrained")
for j, ax in enumerate(axes.flat):
    if j >= len(FEATURES):
        ax.axis("off")
        continue
    im = ax.imshow(np.clip(zr[:, :, j], -8, 8).T, aspect="auto", cmap=cmap, vmin=-8, vmax=8, interpolation="nearest",
                   extent=[minutes[0] - STEP / 60, minutes[-1], len(SERVICES) - .5, -.5])
    ax.axvline(0, color="k", lw=.8)
    ax.set_title(FEATURES[j], fontsize=9)
axes[0, 0].set_yticks(range(len(SERVICES)), SERVICES)
for ax in axes[:, 0]:
    ax.tick_params(axis="y", labelsize=6)
fig.colorbar(im, ax=axes, shrink=.5, label="robust z vs the settle (clipped at ±8); grey = no sample")
fig.supxlabel("minutes from the injection (black line = injection)")
fig.suptitle(f"{r.id}: every feature of every service", fontsize=11)
plt.show()

# How to read it: the robust z of the cell that moved most after the injection, step by step
za = np.where((r.t_rel > 0)[:, None, None], np.abs(zr), np.nan)
i, s, j = np.unravel_index(np.nanargmax(za), za.shape)
f, settle = FEATURES[j], r.x[ex.settle_mask(r), s, j]
med, q1, q3 = np.nanmedian(settle), np.nanpercentile(settle, 25), np.nanpercentile(settle, 75)
scale = max(q3 - q1, .01 * abs(med), ex.MIN_SCALE[f])
print(f"Worked example: the cell that moved most after the injection is {SERVICES[s]} / {f} at minute {minutes[i]:.1f}.")
print(f"  settle: median {show(med, f)}, IQR {show(q3 - q1, f)}; scale used {show(scale, f)}"
      f" (the IQR, floored at 1% of the median and at MIN_SCALE = {show(ex.MIN_SCALE[f], f)})")
print(f"  value {show(r.x[i, s, j], f)}  ->  z = (value - median) / scale = {zr[i, s, j]:+.1f}")

pd.DataFrame(r.labels["segments"]).assign(start=lambda d: ((d.start - r.labels["t_inject"]) / 60).round(1)).rename(
    columns={"start": "start (min)", "seconds": "length (s)", "rate": "target rate", "delivered": "delivered (req/s)",
             "p99_ms": "p99 (ms)", "non_2xx": "non-2xx"})

## 2 · Quality

**Question.** Which service × feature cells carry information?

In the clean sessions under load every cell gets one status:

- `absent`: no sample at all. Examples: throttling in Social, which has no CPU limits; page cache in services without
  a disk.
- `flat`: present, but between its 5th and 95th percentile it moves less than the smallest change that matters
  (`explore.MIN_SCALE`, printed below). The model learns nothing there, and a z-score with almost zero spread explodes.
- `live`: everything else.

**What is plotted.**

- The heatmap has one cell per service (row) and feature (column), coloured by status.
- The first table gives the threshold per feature.
- The second table counts the cells per status and feature.

In [ ]:
present = clean[FEATURES].notna().groupby(level="service").mean().reindex(SERVICES)
spread = (clean.groupby(level="service")[FEATURES].quantile(.95)
          - clean.groupby(level="service")[FEATURES].quantile(.05)).reindex(SERVICES)
floor = pd.Series(ex.MIN_SCALE)[FEATURES]
status = pd.DataFrame("live", index=SERVICES, columns=FEATURES)
status = status.mask(spread.lt(floor, axis=1) | spread.isna(), "flat").mask(present == 0, "absent")
alive = status != "absent"

display(pd.Series({f: show(ex.MIN_SCALE[f], f) for f in FEATURES}, name="smallest change that matters").to_frame().T)
fig, ax = plt.subplots(figsize=(9, 1.2 + .3 * len(SERVICES)), layout="constrained")
ax.imshow(status.map({"absent": 0, "flat": 1, "live": 2}.get).to_numpy(dtype=float), aspect="auto",
          cmap=ListedColormap(["#cccccc", "#f2d16b", "#5bb974"]), vmin=0, vmax=2, interpolation="nearest")
ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=60, ha="right", fontsize=7)
ax.set_yticks(range(len(SERVICES)), SERVICES, fontsize=7)
ax.set_xlabel("feature")
ax.set_ylabel("service")
ax.legend(handles=[Patch(color=c, label=l) for c, l in [("#5bb974", "live"), ("#f2d16b", "flat"), ("#cccccc", "absent")]],
          loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
ax.set_title("status of each cell in the clean sessions", fontsize=10)
plt.show()
display(status.apply(lambda c: c.value_counts()).reindex(["live", "flat", "absent"]).fillna(0).astype(int))
print("services with no live cell:", [s for s in SERVICES if not (status.loc[s] == "live").any()] or "none")
print("features flat or absent in every service:", [f for f in FEATURES if not (status[f] == "live").any()] or "none")

**Question.** How many samples are missing, per split and phase?

**What is plotted.** A table:

- Rows: split and phase (`before` = under load up to the injection, `after` = after it).
- Columns: features. Each value is the share of steps without a sample, counted only on cells that exist (not
  `absent`). `mean` is the average over features.

**How to read it.** In the clean sessions and before the injection the share should be close to 0. After the injection
in the test, a missing sample is often the fault itself: a killed pod sends no metrics.

In [ ]:
keep = alive.loc[df.index.get_level_values("service")].to_numpy()
na = np.array(df[FEATURES].isna(), dtype=float)
na[~keep] = np.nan
missing = pd.DataFrame(na, columns=FEATURES).assign(split=df.split.to_numpy(), phase=df.phase.to_numpy())
missing = missing[missing.phase != "off"].groupby(["split", "phase"]).mean()
missing.insert(0, "mean", missing.mean(axis=1))
missing.round(3)

**Question.** Which missing samples are skipped samples, and which mean that something is gone?

**Why samples go missing.**

- **Skipped sample.** cAdvisor takes a sample every 10 to 43 s (median 16 s). When two samples are more than 30 s
  apart, a 30 s step gets none. A skipped sample hits a whole group of one service for one step: either all of its
  container features or all of its network features. The network features are a separate series with their own
  timestamps.
- **Absence.** A pod that is gone (failed, never scheduled, crash-looping, scaled to zero) sends no samples until it
  comes back. That leaves a hole of many steps, often until the end of the run and often on both groups. A restart
  also leaves the rate features empty for a step, because a rate needs two samples of the new container. These holes
  are the fault's signal.
- **Export cut.** The last step of a run can be cut short by the export.

**The rule** (`anomaly.inputs.fill_misses`, the same code the online detector will use). A group is filled at step t
with its value at t−1 when all of these hold:

- the whole group is missing at t;
- the whole group was there at t−1;
- the other group of the service is there at t;
- the service did not restart at t−1 or t.

Everything else stays NaN, and the group counts as absent (`present = False`). The rule looks only at t−1 and t, like
the online detector, so a long hole has its first step filled and becomes absence from the second step on.

**What is printed.**

- **A table per split, fault run or not, and phase**: missing cells (on the cells that exist in the clean sessions)
  before and after filling, the share filled, and the absence events per hour. An absence event is one group of one
  service absent for one step. The first step of each run is left out: no rate exists there yet.
- **The fault runs with absence after the injection**:
  - share of the steps after the injection where the target is absent;
  - minute of the first absence;
  - other services absent.
- **A worked example**: one filled gap in a clean session.

In [ ]:
from anomaly.inputs import fill_misses

CONT = [j for j, f in enumerate(build.FEATURES) if f[3] == "container"]
NET = [j for j, f in enumerate(build.FEATURES) if f[3] == "pod"]
exists = present.to_numpy() > 0  # the cells of the clean sessions, as the online detector would use them
group_exists = np.stack([exists[:, CONT].any(axis=1), exists[:, NET].any(axis=1)], axis=1)  # [S, 2]
filled, rows, after_rows = {}, [], []
for k, r in runs.items():
    y, pr = fill_misses(r.x.astype(float), FEATURES, exists)
    filled[k] = (y, pr)
    on = (r.rate >= 0) & (r.t > r.labels["t_workload"] + STEP)  # the first step has no rate yet
    fault = r.labels["fault_type"] != "No fault"
    absent = ~pr & group_exists[None]  # [T, S, 2]
    for phase, sel in (("before", on & (r.t_rel <= 0)), ("after", on & (r.t_rel > 0))):
        if sel.any():
            cells = np.broadcast_to(exists, r.x[sel].shape)
            rows.append({"split": r.split, "fault run": fault, "phase": phase,
                         "missing cells": np.isnan(r.x[sel])[cells].sum(), "after filling": np.isnan(y[sel])[cells].sum(),
                         "absence events": absent[sel].sum(), "hours": sel.sum() * STEP / 3600})
    sel = on & (r.t_rel > 0)
    if fault and absent[sel].any():
        targets = [SERVICES.index(t) for t in ex.target_services(r)]
        t_abs = absent[:, targets].any(axis=(1, 2)) & sel
        after_rows.append({"run": k, "target": ", ".join(SERVICES[i] for i in targets),
                           "target absent (share of steps)": t_abs.sum() / sel.sum(),
                           "first absent (min)": r.t_rel[t_abs][0] / 60 if t_abs.any() else np.nan,
                           "other services absent": ", ".join(SERVICES[i] for i in np.nonzero(absent[sel].any(axis=(0, 2)))[0]
                                                              if i not in targets) or "-"})
gaps = pd.DataFrame(rows).groupby(["split", "fault run", "phase"]).sum()
gaps.insert(2, "filled", 1 - gaps["after filling"] / gaps["missing cells"].where(gaps["missing cells"] > 0))
gaps["absence per hour"] = gaps["absence events"] / gaps["hours"]
display(gaps.round(2))
display(pd.DataFrame(after_rows).set_index("run").round(2) if after_rows else "no absence after any injection")

# worked example: the first gap filled in a clean session
for k in clean_ids:
    y, pr = filled[k]
    hit = np.argwhere(np.isnan(runs[k].x[:, :, CONT]).all(axis=2) & ~np.isnan(y[:, :, CONT[0]]))
    if len(hit):
        t, s = hit[0]
        f = "mem_working_set"
        v = runs[k].x[t - 1:t + 2, s, FEATURES.index(f)]
        print(f"Worked example: {k}, {SERVICES[s]}, minute {runs[k].t_rel[t] / 60:.1f}. The container group has no sample"
              f" at that step; {f} is {show(v[0], f)} before and {show(v[2], f)} after, so the step is filled with"
              f" {show(y[t, s, FEATURES.index(f)], f)}, the value before. The network group was there, so this is a"
              f" skipped sample, not a gone pod.")
        break

**Question.** What shape do the distributions have?

**What is plotted.**

- **The table** covers the live cells of the clean sessions, one row per feature:
  - `live cells`: how many services have this feature live.
  - `decades between services`: log10 of the busiest service's median divided by the quietest one's. 2 means the
    busiest service is 100 times the quietest.
  - `skew raw` and `skew after prep`: the skewness, the median over services, before and after `prep`. 0 means
    symmetric; above 1, a long right tail (a few very large values); below −1, a long left tail.
- **The box plots** show four features, one box per service, over the 30 s steps of the clean sessions:
  - box = 25th to 75th percentile, line inside = median;
  - whiskers = the furthest values within 1.5 IQR of the box; outliers are hidden;
  - the x-axis is logarithmic, so services that differ by 100 times fit side by side.

In [ ]:
def skew(v):
    v = v[~np.isnan(v)]
    s = v.std() if len(v) else 0
    return np.nan if len(v) < 20 or s == 0 else float(((v - v.mean()) ** 3).mean() / s**3)


Xc = np.concatenate([runs[k].x[runs[k].rate >= 0] for k in clean_ids])
Pc = np.concatenate([P[k][runs[k].rate >= 0] for k in clean_ids])
rows = []
for j, f in enumerate(FEATURES):
    live = [s for s in range(len(SERVICES)) if status.iloc[s, j] == "live"]
    if not live:
        continue
    meds = np.nanmedian(Xc[:, live, j], axis=0)
    pos = meds[meds > 0]
    rows.append({"feature": f, "live cells": len(live), "decades between services": np.log10(pos.max() / pos.min()),
                 "skew raw": np.nanmedian([skew(Xc[:, s, j]) for s in live]),
                 "skew after prep": np.nanmedian([skew(Pc[:, s, j]) for s in live])})
display(pd.DataFrame(rows).set_index("feature").round(2))

SHOW = {"cpu_user": "cpu_user (mC)", "mem_working_set": "mem_working_set (MiB)", "rx_bytes": "rx_bytes (B/s)",
        "threads": "threads (count)"}
fig, axes = plt.subplots(1, len(SHOW), figsize=(16, 1.2 + .3 * len(SERVICES)), sharey=True, layout="constrained")
for ax, (f, label) in zip(axes, SHOW.items()):
    j = FEATURES.index(f)
    vals = [Xc[:, s, j][~np.isnan(Xc[:, s, j])] * UNIT.get(f, 1) for s in range(len(SERVICES))]
    ax.boxplot([v[v > 0] if (v > 0).any() else [np.nan] for v in vals], orientation="horizontal", showfliers=False)
    ax.set_xscale("log")
    ax.set_xlabel(label + ", log scale")
    ax.set_title(f, fontsize=9)
axes[0].set_yticks(range(1, len(SERVICES) + 1), SERVICES, fontsize=7)
fig.suptitle("values per service in the clean sessions (box = middle half, line = median)", fontsize=10)
plt.show()

### Reading

- **Hotel.** CPU and network are live in 12 of 17 services. The MongoDBs of geo, profile, rate, recommendation and user are almost idle, because their data are loaded at start-up, so their CPU and network are flat. `cpu_throttled` is flat everywhere: healthy sessions never reach the 1000m limit. `mem_cache` is live in one service only.
- **Social.** `cpu_throttled` is absent, since there are no CPU limits. media-frontend, media-memcached and url-shorten-memcached have no live cell, because the load never reaches them. The memory of the stateless services (text, media, unique-id, user-mention, url-shorten) is flat.
- **Both.**
  - `restarts` is flat by construction: check V2 keeps it at 0 in training.
  - Missing samples: 0.1-0.3% in the clean sessions and before the injection; 1.5-2% after the injection in the test, where faults stop pods.
  - The network spans 2-4 decades between services (4.2 for `rx_bytes` in Social), with skew 1.4-1.6. After `prep`, CPU and network have \|skew\| below 0.5.
  - Memory and threads are not skewed to begin with (skew −1.2 to 0.9), and `log1p` makes them skewed to the left.
- **Missing samples.**
  - In the clean sessions the rule fills 97% (Hotel) and 100% (Social) of the missing cells. What is left in Hotel is one service at the last step of one session, cut by the export. Online only complete steps are scored, so this cannot happen there.
  - Absence events in normal steps: 0.11/h in Hotel (that one step) and 0 in Social. Without the rule, the skipped samples would be about 3/h and 8/h.
  - After the injection the target becomes absent for 81-100% of the steps, from 0.3-1.3 minutes on, in these runs:
    - Hotel: misconfig_app, pod_failure, revoke_auth and user_unregistered (the target's app container stops reporting);
    - Social: assign_to_non_existent_node, scale_pod_zero and 4 of the 5 auth_miss_mongoDB runs.
  - Not absence: container_kill, where the container is back within a step and the restart channel carries the fault. Also Social's pod_failure on user-timeline-mongodb: the container keeps reporting while it runs Chaos Mesh's pause image, so its memory drops from 60 to 0.6 MiB and its CPU to 0. That is a collapse of the values, not a gap.
- **For the model.**
  - Give the model the filled values plus `present`. Mask the reconstruction loss where values are absent, and let `present` carry the absence into the score.
  - Keep `restarts`, with a floor on its scale: any restart is anomalous, but a z-score with zero spread explodes.
  - Missing samples after a fault are signal: mask them, do not impute them.
  - Apply `log1p` to CPU and network only.

## 3 · Load

**Question.** Which features follow the load?

**What is plotted.**

- The heatmap has services as rows and features as columns. Each cell (number and colour) is Spearman ρ between that
  feature and the wrk2 rate, over the steady steps of the clean sessions.
  - Red: the feature rises with the load. Blue: it falls. White: unrelated.
  - Blank cells are not live (section 2).
- A feature close to 1 moves with the traffic: the model has to learn that relation, or it will mistake a load
  increase for a fault.
- The table below the heatmap is the median over services.

In [ ]:
def spearman(a, b):
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank()) if ok.sum() > 20 and a[ok].nunique() > 1 else np.nan


rho = steady_clean.groupby(level="service").apply(
    lambda d: pd.Series({f: spearman(d[f], d.rate) for f in FEATURES})).reindex(SERVICES)
rho = rho.where(status == "live")
heat(rho, "how much each feature follows the load (clean sessions, steady steps)", cbar="Spearman ρ with the wrk2 rate")
plt.show()
display(rho.median().round(2).to_frame("median over services").T)

print("How to read it.")
extremes(rho)
print("share of live cells with ρ > 0.8:", (rho.gt(.8).sum() / rho.notna().sum()).dropna().round(2).to_dict())
s0 = rho["cpu_user"].idxmax()
d = steady_clean.xs(s0, level="service")
lo_r, hi_r = d.rate.min(), d.rate.max()
print(f"Worked example: {s0} / cpu_user has ρ = {rho.loc[s0, 'cpu_user']:.2f}. Its median is"
      f" {show(d[d.rate == lo_r].cpu_user.median(), 'cpu_user')} at {lo_r} req/s and"
      f" {show(d[d.rate == hi_r].cpu_user.median(), 'cpu_user')} at {hi_r} req/s.")

**Question.** How does a feature respond to the load, and how much does the mix matter?

**What is plotted.**

- One panel per service where `FEATURE_LOAD` is live.
- x = wrk2 rate (req/s). y = median of `FEATURE_LOAD` at that rate over the steady clean steps, in the unit on the axis.
- One line per mix; the legend names them (table in section 0).
- Both axes are logarithmic. A straight line then means a power law, and its slope says how the feature scales:
  - slope 1: proportional to the rate (twice the requests, twice the CPU);
  - slope 0: flat.
- Parallel lines: the mix only shifts the level. Lines with different slopes, or crossing: the mix changes the work
  the service does.

The printed slope is a straight-line fit of log(feature) against log(rate), all mixes pooled.

In [ ]:
FEATURE_LOAD = "cpu_user"
unit_label = {"cpu": "mC", "mem": "MiB", "thr": "count", "res": "per 5 min"}.get(FEATURE_LOAD[:3], "per s")
lev = steady_clean.groupby([steady_clean.index.get_level_values("service"), "mix", "rate"])[FEATURE_LOAD].median()
live = [s for s in SERVICES if status.loc[s, FEATURE_LOAD] == "live"]
ncol = 6
nrow = -(-len(live) // ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 2.5 * nrow), squeeze=False, layout="constrained")
slopes = {}
for ax, s in zip(axes.flat, live):
    for m, d in lev.loc[s].groupby(level="mix"):
        d = d.droplevel("mix")
        ax.plot(d.index, d.to_numpy() * UNIT.get(FEATURE_LOAD, 1), "o-", ms=3, label=f"mix {m}: {MIXES[m][0]}")
    pts = lev.loc[s].reset_index()
    pts = pts[pts[FEATURE_LOAD] > 0]
    slopes[s] = np.polyfit(np.log(pts.rate), np.log(pts[FEATURE_LOAD]), 1)[0] if len(pts) > 3 else np.nan
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"{s} (slope {slopes[s]:.2f})", fontsize=8)
    ax.tick_params(labelsize=6)
for ax in axes.flat[len(live):]:
    ax.axis("off")
axes[0, 0].legend(fontsize=6)
fig.supxlabel("wrk2 rate (req/s, log scale)")
fig.supylabel(f"median {FEATURE_LOAD} ({unit_label}, log scale)")
fig.suptitle(f"{FEATURE_LOAD} against the load, one line per request mix (clean sessions, steady steps)", fontsize=10)
plt.show()
print("How to read it: log-log slope per service (1 = proportional to the rate):")
print(pd.Series(slopes).round(2).sort_values().to_dict())

### Reading

- **Both.** CPU and network follow the load in almost every live service: median ρ 0.93-0.96, and 81-91% of their live cells above 0.8. Memory and threads barely do (0.10-0.37).
- **Hotel.**
  - The application services scale almost in proportion: the log-log slope of `cpu_user` is 0.86-0.93. The memcached caches and `user` scale less (0.60-0.78), mongodb-reservation 0.50.
  - The bookings mix multiplies the CPU of `user` 3 to 10 times and raises mongodb-reservation. The search-heavy mix lowers recommendation.
- **Social.**
  - The application services scale with slope 0.84-0.92, the stores with 0.45-0.70.
  - The mixes with timeline reads (1, 2) lower the compose path (compose-post, text, user, unique-id, media, user-mention, url-shorten) and raise nginx-thrift and post-storage.
  - post-storage-memcached works only when timelines are read: it is flat under mix 0.
- **For the model.** The load confound lives in 8 of the 13 features, which move together, so the model can learn it from their relations without being given the load. The clean sessions cover every mix in every rate band (next section), so the model also sees the mix effects.

## 4 · Training against test

**Question.** Do the clean sessions cover what the model will see? First, the load.

**What is plotted.** A table:

- Rows: bands of the wrk2 rate in req/s. `(15, 30]` means above 15, up to 30.
- Columns: split, then mix.
- Values: minutes of load spent in that band.

**How to read it.** A zero under `clean` where `dev` or `test` has minutes would be a situation the model never trains
on. The check below prints any such case.

In [ ]:
steps = df.xs(SERVICES[0], level="service")[["rate", "mix", "split", "phase"]]
steps = steps[steps.rate >= 0]
cov = (pd.crosstab(pd.cut(steps.rate, [0, 15, 30, 60, 120, 200, 400]), [steps.split, steps.mix]) * STEP / 60).round().astype(int)
cov.index.name = "rate band (req/s)"
display(cov)
gaps = [(band, mix) for band in cov.index for (split, mix) in cov.columns
        if split != "clean" and cov.loc[band, (split, mix)] > 0 and cov.loc[band].get(("clean", mix), 0) == 0]
print("bands and mixes seen in dev/test but never in clean:", sorted(set(gaps)) or "none")

**Question.** How many normal steps of dev and test fall outside what the clean sessions showed?

**Definition.**

- The range of a cell spans its 0.5th to 99.5th percentile in the clean sessions, widened on each side by the
  smallest change that matters (`MIN_SCALE`).
- A **normal** step lies before the injection, or anywhere in a `noop` run. A normal step out of range is a likely
  false alarm with any model: training never saw that value.
- After the injection in a fault run, leaving the range is the signal.

**What is plotted.**

- **The table**: share of steps out of range, averaged over the existing cells, per split, fault run or not, and
  phase.
- **The heatmap**: rows services, columns features. Colour and number give the share of the normal dev and test steps
  out of range for that cell. 0 means always inside, 0.3 means 30% outside.

In [ ]:
svc = df.index.get_level_values("service")
lo = clean.groupby(level="service")[FEATURES].quantile(.005).reindex(SERVICES) - floor
hi = clean.groupby(level="service")[FEATURES].quantile(.995).reindex(SERVICES) + floor
out = np.array(df[FEATURES].lt(lo.loc[svc].to_numpy()) | df[FEATURES].gt(hi.loc[svc].to_numpy()), dtype=float)
out[~keep | df[FEATURES].isna().to_numpy()] = np.nan
fault = info.fault.reindex(df.index.get_level_values("run")).ne("No fault").to_numpy()
out = pd.DataFrame(out, columns=FEATURES, index=df.index).assign(split=df.split.to_numpy(), phase=df.phase.to_numpy(),
                                                                 fault_run=fault)
out = out[out.phase != "off"]
display(out.groupby(["split", "fault_run", "phase"])[FEATURES].mean().mean(axis=1).round(3).to_frame("share out of range"))

normal = out[(out.split != "clean") & (~out.fault_run | (out.phase == "before"))]
risk = normal.groupby(level="service")[FEATURES].mean().reindex(SERVICES)
heat(risk, "normal dev and test steps outside the clean range", cbar="share of normal steps out of range",
     vmin=0, vmax=.5, cmap="Reds")
plt.show()

print("How to read it.")
extremes(risk)
(s, f), share = risk.stack().idxmax(), risk.stack().max()
vals = df.loc[normal.index, f].xs(s, level="service")  # the values behind the out-of-range flags
outside = vals[(vals < lo.loc[s, f]) | (vals > hi.loc[s, f])]
print(f"Worked example: {s} / {f}. Its clean range is {show(lo.loc[s, f] + floor[f], f)} to"
      f" {show(hi.loc[s, f] - floor[f], f)} (widened by {show(floor[f], f)} per side). {share:.0%} of its normal steps fall"
      f" outside, with median {show(outside.median(), f)}.")

**Question.** Do sessions differ from each other more than a session changes over time?

**What is plotted.**

- Rows: services. Columns: features. Only load levels (rate, mix) seen by at least 3 clean sessions count.
- At each such level the score is log10 of the between-session variance (the variance of the session medians) divided
  by the within-session variance (the mean variance of the steps inside each session). The cell shows the median over
  levels:
  - 0: sessions differ as much as steps inside a session;
  - +1: the between-session variance is 10 times larger, so each session sits at a level of its own;
  - −1: 10 times smaller, so the sessions agree.
- Blank = not live.

**Why it matters.** A level of its own per session, standardised with global statistics, looks like an anomaly in a
new session.

In [ ]:
sc = steady_clean.reset_index()
g = sc.groupby(["service", "rate", "mix", "run"])[FEATURES]
med, var = g.median(), g.var()
lvl = ["service", "rate", "mix"]
n = med.groupby(level=lvl).size()
ratio = (med.groupby(level=lvl).var() / var.groupby(level=lvl).mean())[n >= 3]
ratio = np.log10(ratio.replace([np.inf, 0], np.nan)).groupby(level="service").median().reindex(SERVICES)
ratio = ratio.where(status == "live")
heat(ratio, "between-session against within-session variance at the same load",
     cbar="log10(between / within); +1 = 10x larger", vmin=-2, vmax=2)
plt.show()
display(ratio.median().round(2).to_frame("median over services").T)

print("How to read it.")
extremes(ratio)
s, f = ratio.stack().idxmax()
m = med.loc[s][f].dropna()
level = m.groupby(level=["rate", "mix"]).size().idxmax()
per_session = m.loc[level]
print(f"Worked example: {s} / {f} at rate {level[0]}, mix {level[1]}: the session medians go from"
      f" {show(per_session.min(), f)} to {show(per_session.max(), f)}, while inside a session the typical standard"
      f" deviation is {show(np.sqrt(var.loc[s].loc[level][f].mean()), f)}.")

**Question.** Which memories grow during a session?

**What is plotted.**

- **The table**: the slope of the working-set memory in each clean session, from a straight-line fit, in MiB per hour.
  It gives the median, minimum and maximum over the 12 sessions, sorted by the size of the median.
- **The charts**: the three fastest-growing memories.
  - x = minutes since the load started; y = working-set memory (MiB).
  - One coloured line per clean session; the colour is the session's mean rate (colour bar).
  - Black dashed lines: the `noop` runs of the test. They last about 20 minutes, so they cover only the left part.

In [ ]:
def slope_mib_h(r, s, f="mem_working_set"):
    y = r.x[:, SERVICES.index(s), FEATURES.index(f)] / 2**20
    ok = ~np.isnan(y) & (r.rate >= 0)
    return np.polyfit((r.t[ok] - r.t[ok][0]) / 3600, y[ok], 1)[0] if ok.sum() > 20 else np.nan


mem_slopes = pd.DataFrame({k: {s: slope_mib_h(runs[k], s) for s in SERVICES} for k in clean_ids})
growth = pd.DataFrame({"median": mem_slopes.median(axis=1), "min": mem_slopes.min(axis=1), "max": mem_slopes.max(axis=1)})
growth = growth.reindex(growth["median"].abs().sort_values(ascending=False).index)
growth.columns = [f"{c} (MiB/h)" for c in growth.columns]
display(growth.head(8).round(1))

STORES = list(growth.index[:3])
noop = [k for k in runs if k.startswith("test/") and info.loc[k, "fault"] == "No fault"]
mean_rate = {k: runs[k].rate[runs[k].rate >= 0].mean() for k in clean_ids}
norm = plt.Normalize(min(mean_rate.values()), max(mean_rate.values()))
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4), layout="constrained")
for ax, s in zip(axes, STORES):
    for k in clean_ids + noop:
        r = runs[k]
        y = r.x[:, SERVICES.index(s), FEATURES.index("mem_working_set")] / 2**20
        x = (r.t - r.labels["t_workload"]) / 60
        style = dict(color=plt.cm.viridis(norm(mean_rate[k])), lw=1) if k in mean_rate else dict(color="k", ls="--", lw=1)
        ax.plot(x, y, **style)
    ax.set_title(s, fontsize=9)
    ax.set_xlabel("minutes since the load started")
axes[0].set_ylabel("working-set memory (MiB)")
axes[-1].legend(handles=[Line2D([], [], color="k", ls="--", label="test noop runs")], fontsize=7, loc="upper left")
fig.colorbar(plt.cm.ScalarMappable(norm, "viridis"), ax=axes, label="mean rate of the clean session (req/s)")
plt.show()

**Question.** Should memory be given as a level or as a change?

**What is plotted.** For the same three memories, a table over the clean sessions after the settle. Each column is a
Pearson correlation:

- `level ~ rate`: memory level against the wrk2 rate.
- `change ~ rate`: the change per step against the rate. The change is a 2-minute moving average, in MiB per 30 s.
- `change ~ own rx_packets`: the change against the packets the store itself receives.

**How to read it.** If the change follows the store's own traffic and the level does not, then the level depends on
the load history, which a window of a few minutes cannot see. The change, instead, is explained by what is in the
window. That is the case for `MEM_DELTA = True`.

In [ ]:
def corr(a, b):
    ok = ~np.isnan(a) & ~np.isnan(b)
    return np.corrcoef(a[ok], b[ok])[0, 1] if ok.sum() > 20 and a[ok].std() > 0 and b[ok].std() > 0 else np.nan


rows = []
for s in STORES:
    lv, d, rt, rx = [], [], [], []
    for k in clean_ids:
        r = runs[k]
        on = (r.t > r.labels["t_workload"] + 300) & (r.rate >= 0)
        y = r.x[:, SERVICES.index(s), FEATURES.index("mem_working_set")] / 2**20
        dy = np.convolve(np.diff(y, prepend=np.nan), np.ones(4) / 4, "same")
        lv += list(y[on]); d += list(dy[on]); rt += list(r.rate[on])
        rx += list(r.x[on, SERVICES.index(s), FEATURES.index("rx_packets")])
    lv, d, rt, rx = map(np.array, (lv, d, rt, rx))
    rows.append({"service": s, "level ~ rate": corr(lv, rt.astype(float)), "change ~ rate": corr(d, rt.astype(float)),
                 "change ~ own rx_packets": corr(d, rx)})
pd.DataFrame(rows).set_index("service").round(2)

### Reading

- **Both.**
  - The load is covered: the clean sessions reach 240 req/s and dev and test 160, with the same mixes.
  - Out of range: 0.1% of the clean steps after the settle; 0.3% (Hotel) and 0.5% (Social) of the normal test steps; 6% (Hotel) and 11% (Social) after a fault.
  - The `before` phase is at 2-3% in every split, clean included: this is the first minute of load, not a shift between splits.
  - Between sessions at the same load the network agrees: log10 ratio about −1, ten times below the variation inside a session. Memory sits at a level of its own in each session (log10 ratio 0.7-1, 5 to 10 times above).
- **Hotel.** Memory does not grow: the largest slope is mongodb-reservation, +5 MiB/h. Its change per step adds nothing: the correlations stay below 0.5.
- **Social.**
  - The riskiest cells are the threads of user-service (30% of the normal steps out of range) and social-graph-service (23%). `cpu_user` also has a level per session (log10 ratio 0.63).
  - home-timeline-redis grows by about 880 MiB/h, post-storage and url-shorten mongo by about 380, faster in busier sessions.
  - The level does not follow the rate (r 0.10-0.11). The change per step follows the store's own packets: 0.87-0.89 for the mongos, 0.42 for redis.
- **For the model.** In Social give memory as its change per step (`MEM_DELTA = True`). In Hotel it makes no difference.

## 5 · Structure

**Question.** Which features repeat each other?

First the exact identities, then the correlation between features.

**What is plotted.**

- The heatmap has features as both rows and columns. Each cell is Spearman ρ between two features of the same service
  over the steady clean steps, the median over services.
- A high correlation in healthy data does not make a feature useless: it is exactly the relation the model learns,
  and a fault breaks it. Examples: CPU rising without traffic, or packets arriving with no CPU behind them.
- Dropping a feature makes sense only when it is an exact identity of the others.

In [ ]:
c = clean[["cpu", "cpu_user", "cpu_system"]].dropna()
print(f"cpu - (cpu_user + cpu_system): median |residual| {(c.cpu - c.cpu_user - c.cpu_system).abs().median():.1e} cores,"
      f" against a median cpu of {c.cpu.median():.1e} cores")
ranks = steady_clean[FEATURES].groupby(level="service").rank()
fcorr = ranks.groupby(level="service").corr().groupby(level=1).median().reindex(index=FEATURES, columns=FEATURES)
heat(fcorr, "correlation between features of the same service (median over services)",
     cbar="Spearman ρ", xlabel="feature", ylabel="feature")
plt.show()

**Question.** How many dimensions do a service's data really have?

**Method.** PCA per service on the `Z` values of the clean sessions, using the live features without `DROP`. PCA
rotates the features into independent directions (components), sorted by how much of the variance they explain.

**What is plotted.**

- **The table**, per service:
  - number of live features;
  - `k90`, `k95`, `k99`: components needed for 90, 95 and 99% of the variance;
  - `var PC1`: share of the variance on the first component;
  - `|corr(PC1, rate)|`: how closely the first component follows the load.
- **The chart**:
  - x = number of components; y = cumulative share of variance explained;
  - one line per service, coloured by type;
  - dotted line = 95%.

**Why it matters.** It is the starting point for the latent size. A service with k95 = 2 does not need a latent of 16
per step.

In [ ]:
DROP = ["cpu"]  # = cpu_user + cpu_system
TYPE_COLOR = {"Service": "tab:blue", "Database": "tab:green", "Cache": "tab:orange"}
rows, curves, loadings = [], {}, {}
for s, name in enumerate(SERVICES):
    cols = [j for j, f in enumerate(FEATURES) if f not in DROP and status.loc[name, f] == "live"]
    if len(cols) < 2:
        continue
    v = np.concatenate([Z[k][runs[k].rate >= 0][:, s, cols] for k in clean_ids])
    rt = np.concatenate([runs[k].rate[runs[k].rate >= 0] for k in clean_ids]).astype(float)
    ok = ~np.isnan(v).any(axis=1)
    v, rt = v[ok] - v[ok].mean(axis=0), rt[ok]
    u, sv, vt = np.linalg.svd(v, full_matrices=False)
    ev = np.cumsum(sv**2) / (sv**2).sum()
    curves[name] = ev
    loadings[name] = pd.Series(vt[0], index=[FEATURES[j] for j in cols])
    rows.append({"service": name, "type": NODE_TYPE[name], "live features": len(cols),
                 "k90": int(np.searchsorted(ev, .90) + 1), "k95": int(np.searchsorted(ev, .95) + 1),
                 "k99": int(np.searchsorted(ev, .99) + 1), "var PC1": ev[0], "|corr(PC1, rate)|": abs(corr(u[:, 0], rt))})
pca = pd.DataFrame(rows).set_index("service")
display(pca.round(2))

fig, ax = plt.subplots(figsize=(8, 3.8), layout="constrained")
for name, ev in curves.items():
    ax.plot(range(1, len(ev) + 1), ev, "o-", ms=2, lw=1, color=TYPE_COLOR[NODE_TYPE[name]], alpha=.8)
ax.axhline(.95, color="k", ls=":")
ax.legend(handles=[Line2D([], [], color=c, label=t) for t, c in TYPE_COLOR.items()]
          + [Line2D([], [], color="k", ls=":", label="95% of the variance")], fontsize=8)
ax.set_xlabel("number of components")
ax.set_ylabel("cumulative share of variance")
ax.set_title("variance explained per service (one line per service)", fontsize=10)
plt.show()

name = pca["live features"].idxmax()
w = loadings[name].sort_values(key=abs, ascending=False)
print(f"Worked example: {name} has {pca.loc[name, 'live features']} live features but k95 = {pca.loc[name, 'k95']}."
      f" Its first component explains {pca.loc[name, 'var PC1']:.0%} of the variance and has |corr| with the rate"
      f" {pca.loc[name, '|corr(PC1, rate)|']:.2f}. Its weights (all features move together when they have the same"
      f" sign): {w.round(2).to_dict()}")

**Question.** How long must the window W be?

**What is plotted.** The autocorrelation of the `Z` values in the clean sessions, at lags of 1 to 20 steps (30 s to 10
minutes).

- x = lag in minutes; y = autocorrelation.
- One line per feature: the median over live cells and sessions.
- Dotted line = 0.2: below it the signal keeps little memory of its past.
- **Left**: the values as they are. They are dominated by the load segments: while the load is constant, the value
  stays similar.
- **Right**: the residual after subtracting the median of each load segment. This is the signals' own dynamics, with
  the load removed.

The table at the bottom gives how long load segments last (ramp steps of 30 s excluded).

**Why it matters.** The window should span the time over which the signals still carry memory, and no more.

In [ ]:
LAGS = np.arange(1, 21)


def acf(y):
    return [corr(y[:-k], y[k:]) for k in LAGS]


def segment_residual(r, v):
    changed = np.r_[True, (np.diff(r.rate) != 0) | (np.diff(r.mix) != 0)]
    seg, res = np.cumsum(changed), np.full_like(v, np.nan)
    for i in np.unique(seg):
        on = seg == i
        if on.sum() >= 3:
            res[on] = v[on] - np.nanmedian(v[on], axis=0)
    return res


A, R = {f: [] for f in FEATURES}, {f: [] for f in FEATURES}
for k in clean_ids:
    r = runs[k]
    v = np.where((r.rate >= 0)[:, None, None], Z[k], np.nan)
    res = segment_residual(r, v)
    for j, f in enumerate(FEATURES):
        for s, name in enumerate(SERVICES):
            if status.loc[name, f] == "live":
                A[f].append(acf(v[:, s, j]))
                R[f].append(acf(res[:, s, j]))
fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True, layout="constrained")
for ax, data, title in zip(axes, (A, R), ("Z values as they are", "residual inside each load segment")):
    for f in FEATURES:
        if data[f]:
            ax.plot(LAGS * STEP / 60, np.nanmedian(data[f], axis=0), "o-", ms=2, lw=1, label=f)
    ax.axhline(.2, color="k", ls=":", label="0.2: little memory left")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("lag (minutes)")
axes[0].set_ylabel("autocorrelation (median over cells)")
axes[1].legend(fontsize=7, ncol=2)
plt.show()

def below(data):
    # minutes until the median autocorrelation first drops below 0.2; None if it never does within 10 minutes
    out = {}
    for f in FEATURES:
        if data[f]:
            under = np.nanmedian(data[f], axis=0) < .2
            out[f] = float(LAGS[np.argmax(under)] * STEP / 60) if under.any() else None
    return out


print("How to read it: minutes until the autocorrelation drops below 0.2 (None = not within 10 minutes)")
print("  values as they are:", below(A))
print("  residual in segment:", below(R))
seg_min = pd.Series([s["seconds"] for k in clean_ids for s in runs[k].labels["segments"] if s["seconds"] > 30]) / 60
seg_min.describe().round(1).to_frame("load segment length (minutes)").T

**Question.** Which services move together?

**What is plotted.**

- Rows and columns are the services where `rx_packets` is live.
- Each cell is the Pearson correlation of their `rx_packets` (`Z` values) over the clean steps.
- A red block = a group of services whose traffic rises and falls together, usually a call chain.

**Why it matters.** These services light up together even when the fault is in only one of them: they are the
candidates the model has to tell apart for localization.

In [ ]:
j = FEATURES.index("rx_packets")
live = [s for s in SERVICES if status.loc[s, "rx_packets"] == "live"]
rx = pd.DataFrame(np.concatenate([Z[k][runs[k].rate >= 0][:, :, j] for k in clean_ids]), columns=SERVICES)[live]
rxc = rx.corr()
heat(rxc, "how the received traffic of services moves together (clean sessions)", cbar="Pearson r of rx_packets",
     annot=len(live) <= 18, xlabel="service", ylabel="service")
plt.show()
pairs = rxc.where(np.triu(np.ones(rxc.shape, dtype=bool), 1)).stack().dropna().sort_values()
print("How to read it: most correlated pairs:", pairs.tail(5)[::-1].round(2).to_dict())
print("least correlated pairs:", pairs.head(3).round(2).to_dict())

### Reading

- **Both.**
  - `cpu` equals `cpu_user + cpu_system` exactly: the residual is 1e-8 cores against a median of 5-8 mC.
  - The other high correlations (network features 0.99-1.00, CPU with network 0.93-0.97) are exactly what faults break.
  - k95 is 2-3 for Hotel's application services and 1-4 in Social. The first component carries 53-95% of the variance and is the load (\|r\| with the rate 0.5-0.86), except in databases without traffic.
  - As they are, CPU and network stay autocorrelated for 5-5.5 minutes, the typical length of a load segment (median 5 minutes, 2 to 8). Memory stays above 0.2 over the whole 10 minutes: that is its drift.
  - Inside a segment, CPU and network are white noise after 30 s; memory loses its memory in 1-2 minutes.
- **For the model.**
  - Drop `cpu` and keep the rest.
  - Per service and step, 2-4 latent dimensions are enough; for a window, choose the latent size on dev, starting small.
  - The dynamics inside a window is almost all load: W = 10 (5 minutes) spans a segment and its changes, and a longer window adds no dynamics of its own.

## 6 · Faults

**Question.** Is the fault visible, and does the faulty service stand out?

**The detector.** A naive one with no model, `explore.score_services`:

- Each service's score is its largest |robust z| over all features in the `AFTER_S` seconds after the injection. The
  reference is the run's own settle.
- Services are sorted by score. `rank` is the position of the target (1 = first).

It says how visible each fault is in the metrics alone, and it sets a bar for the CMVAE. `dev` is the default; look at
`test` only for the final evaluation.

**What is plotted: the first table**, one row per fault run of `SPLIT`:

- the rate at the injection and the target;
- the target's rank and score;
- the three highest services, each with the feature that gives its score and the score.

The second table summarises each scenario: median rank, and share of runs with the target first (`top1`) or in the
first three (`top3`).

In [ ]:
AFTER_S = 300
rows = []
for k, r in runs.items():
    if r.split != SPLIT or r.labels["fault_type"] == "No fault":
        continue
    scored = ex.score_services(r, after_s=AFTER_S)
    names = [s for s, *_ in scored]
    targets = ex.target_services(r)
    rank = min((names.index(t) + 1 for t in targets), default=None)
    rows.append({"run": k, "scenario": info.loc[k, "scenario"], "rate_inj": info.loc[k, "rate_inj"],
                 "target": ", ".join(targets), "rank": rank,
                 "target score": scored[rank - 1][1] if rank else np.nan,
                 "target feature": scored[rank - 1][2] if rank else "-",
                 "top 3 (feature, score)": ", ".join(f"{s} ({f}, {v:.0f})" for s, v, f in scored[:3])})
faults = pd.DataFrame(rows).set_index("run")
display(faults.round(1))

k0 = faults.index[0]
row = faults.loc[k0]
print(f"Worked example: in {k0} the target {row.target} scores {row['target score']:.0f} (its |z| on"
      f" {row['target feature']} in the {AFTER_S} s after the injection). {row['rank'] - 1} services score higher,"
      f" so its rank is {row['rank']}.")
faults.groupby("scenario").agg(runs=("rank", "size"), median_rank=("rank", "median"),
                               top1=("rank", lambda v: (v == 1).mean()), top3=("rank", lambda v: (v <= 3).mean()))

**Question.** How high does the same score go without a fault?

**What is computed.** The top score over services in the runs without a fault: the clean sessions at their fake
injection (300 s after the load starts) and the `noop` runs. This is the background noise of the detector. A target
scoring below it cannot be told apart from a healthy run with this detector. The table lists the three noisiest
healthy runs, with the service and feature responsible.

With `SPLIT = "test"` a last table gives the rank of the target per scenario (rows) and injection rate (columns).

In [ ]:
quiet = pd.DataFrame([dict(zip(["service", "score", "feature"], ex.score_services(r, after_s=AFTER_S)[0]), run=k)
                      for k, r in runs.items() if r.labels["fault_type"] == "No fault"]).set_index("run")
print("top score without a fault:", quiet.score.describe()[["count", "50%", "max"]].round(1).to_dict())
for name, q in (("the median", quiet.score.median()), ("the maximum", quiet.score.max())):
    print(f"fault runs whose target scores above {name} without a fault: {(faults['target score'] > q).mean():.0%}")
display(quiet.sort_values("score").tail(3).round(1))
if SPLIT == "test":
    display(faults.pivot_table(index="scenario", columns="rate_inj", values="rank").rename_axis(columns="rate at injection"))

**What is plotted.** One panel per fault run, at most `MAX_PLOTS`.

- x = minutes from the injection.
- y = for each service, its largest |robust z| over all features at that step. The y-axis is symlog: linear below 1,
  logarithmic above.
- Coloured line: the target.
- Grey band: 10th to 90th percentile of the other services. Grey dashed line: their maximum.
- Red dotted line: the injection.

**How to read it.** A target line that rises above the dashed line after 0 means the target stands out. A dashed line
that rises as high as the target means another service, or the load, hides it. A target line that stops means the target sends no
samples any more (its pod is gone, as in `assign_to_non_existent_node`). The missing sample is the signal, and this
detector cannot use it.

In [ ]:
MAX_PLOTS = 12
ids = list(faults.index[:MAX_PLOTS])
nrow = -(-len(ids) // 3)
fig, axes = plt.subplots(nrow, 3, figsize=(16, 2.8 * nrow), squeeze=False, layout="constrained")
for ax, k in zip(axes.flat, ids):
    r = runs[k]
    zmax = np.nanmax(np.abs(ex.robust_z(r.x, ex.settle_mask(r))), axis=2)
    targets = [SERVICES.index(t) for t in ex.target_services(r)]
    others = np.delete(zmax, targets, axis=1)
    m = r.t_rel / 60
    ax.fill_between(m, np.nanpercentile(others, 10, axis=1), np.nanpercentile(others, 90, axis=1), color=".85")
    ax.plot(m, np.nanmax(others, axis=1), color=".45", ls="--", lw=.8)
    for t in targets:
        ax.plot(m, zmax[:, t], lw=1.5, label=f"target: {SERVICES[t]}")
    ax.axvline(0, color="red", ls=":")
    ax.set_yscale("symlog", linthresh=1)
    ax.set_ylim(bottom=0)
    ax.set_title(f"{k.split('/')[1]} (rank {faults.loc[k, 'rank']})", fontsize=8)
    ax.legend(fontsize=6, loc="upper left")
for ax in axes.flat[len(ids):]:
    ax.axis("off")
fig.legend(handles=[Patch(color=".85", label="other services, 10th-90th percentile"),
                    Line2D([], [], color=".45", ls="--", label="other services, maximum"),
                    Line2D([], [], color="red", ls=":", label="injection")], loc="outside upper center", ncol=3, fontsize=8)
fig.supxlabel("minutes from the injection")
fig.supylabel("max |robust z| over features (symlog)")
plt.show()

### Reading

- **Hotel (dev).**
  - CPU stress is found: rank 1-3 at every stress level, including 5% (rank 1 and 2).
  - net_delay_search is not found (rank 8): a delay shows on the edge, in the traces, not in the metrics of the target.
- **Social (dev).**
  - CPU stress at 20% and 80% is found (rank 1-3); at 5% it is not (rank 4 and 6).
  - pod_failure of user-timeline-mongodb lands at rank 2 and 13.
  - net_delay_home_timeline lands at rank 1-2, through the threads of home-timeline-service.
- **Both.**
  - The noise is the problem. Without a fault the top score reaches 455 in Hotel: clean sessions whose load rises well above the settle, on the frontend's `rx_bytes`. In Social it reaches 4817: post-storage-memcached, whose traffic grows as its cache fills. Both are more than any target scores.
  - Only 50-62% of the dev targets score above the median healthy run.
- **For the model.** This naive detector is the bar. Its two failures are the two confounds seen above, load and state, and the CMVAE has to learn both from the clean sessions.

## 7 · Free space

Everything is loaded: `runs`, `info`, `df`, `P`, `Z`, `status`, `services_table`, `features_table`, and
`filled` (run id -> the values after `fill_misses` and the `present` flags). A few examples
to start from.

In [ ]:
# one run, one service, some features over time
k = next(k for k in runs if k.startswith("test/"))
axes = df.loc[k].xs(SERVICES[0], level="service")[["cpu_user", "rx_packets", "mem_working_set", "rate"]].plot(
    subplots=True, figsize=(12, 6), title=f"{k}, {SERVICES[0]}")
axes[-1].set_xlabel("seconds from the injection")
plt.show()

# median of a feature per service, split and phase
on = df[df.phase != "off"]
on.groupby([on.index.get_level_values("service"), "split", "phase"])["cpu_user"].median().unstack([1, 2]).head()